In [1]:
import numpy as np

def solve_mars_rover_dp():
    # Parameters from the prompt
    MAX_FU = 10
    TOTAL_OPS = 5
    R_COMP = 10
    T_SAFE = 5
    T_FAST = 2
    PENALTY = 100

    # V[t][f][was_fast] -> max expected reward from state (f, was_fast) at step t
    # t: 0 to 5 (operations completed)
    # f: 0 to 9 (current fatigue units)
    # was_fast: 0 (Safe) or 1 (Fast)
    V = np.zeros((TOTAL_OPS + 1, MAX_FU, 2))

    # Base Case: V[5][f][was_fast] = 0 (Mission ends after 5 ops, no more reward)
    # We iterate backwards from t = 4 down to 0
    for t in range(TOTAL_OPS - 1, -1, -1):
        for f in range(MAX_FU):
            for last_was_fast in [0, 1]:

                # --- ACTION: SAFE ---
                ev_safe = 0
                # +1 FU (0.8), +2 FU (0.2)
                for df, prob in [(1, 0.8), (2, 0.2)]:
                    new_f = f + df
                    if new_f >= MAX_FU:
                        reward = R_COMP - T_SAFE - PENALTY
                        ev_safe += prob * reward
                    else:
                        reward = R_COMP - T_SAFE
                        ev_safe += prob * (reward + V[t+1, new_f, 0])

                # --- ACTION: FAST ---
                ev_fast = 0
                # Determine base increments based on consecutive-fast rule
                base_increments = [(5, 0.6), (7, 0.4)] if last_was_fast == 1 else [(3, 0.7), (4, 0.3)]

                for df_base, p_base in base_increments:
                    # Check for Micro-tear if current fatigue >= 8
                    if f >= 8:
                        # With tear (+4 FU, 0.2) or without (0.8)
                        for df_tear, p_tear in [(4, 0.2), (0, 0.8)]:
                            final_f = f + df_base + df_tear
                            total_p = p_base * p_tear
                            if final_f >= MAX_FU:
                                reward = R_COMP - T_FAST - PENALTY
                                ev_fast += total_p * reward
                            else:
                                reward = R_COMP - T_FAST
                                ev_fast += total_p * (reward + V[t+1, final_f, 1])
                    else:
                        # Standard fast transition (no tear possible)
                        final_f = f + df_base
                        if final_f >= MAX_FU:
                            reward = R_COMP - T_FAST - PENALTY
                            ev_fast += p_base * reward
                        else:
                            reward = R_COMP - T_FAST
                            ev_fast += p_base * (reward + V[t+1, final_f, 1])

                # Bellman Equation: Store the optimal value for this state
                V[t, f, last_was_fast] = max(ev_safe, ev_fast)

    return V

# Run the DP
V_table = solve_mars_rover_dp()

# Output Results
print(f"--- Optimal Value Function Results ---")
print(f"V(0, 0, Safe): {V_table[0, 0, 0]:.2f} (Expected total reward at start)")
print(f"V(4, 4, Safe): {V_table[4, 4, 0]:.2f} (Expected value for state from Q.7.3.4)")


--- Optimal Value Function Results ---
V(0, 0, Safe): 27.43 (Expected total reward at start)
V(4, 4, Safe): 8.00 (Expected value for state from Q.7.3.4)
